#### Setup Cell

Run the setup cell below to import the libraries required to build the LangChain workflow and initialize the language model.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key. 

In [1]:
# Import required libraries
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize the LLM 
llm = ChatOpenAI(
    model="gpt-6-luna"
)

#### Defining Multiple Prompt Templates
 
Use the `PromptTemplate` class to create three different templates, where each template guides the LLM through a distinct task with a corresponding prompt string that includes a placeholder, which is filled at runtime with user-provided input. 

- A `name_prompt` template with a `name` variable that asks the LLM to confirm the user's name 
- A `topic_prompt` template with a `topic` variable that generates a detailed explanation of a topic
- A `summary_prompt` template with an `explanation` variable that produces a short and clear summary of that explanation 

In [2]:
# Define prompt templates
name_prompt = PromptTemplate.from_template(
    "My name is {name}. Remember it!"
)

topic_prompt = PromptTemplate.from_template(
    "Write a detailed paragraph explaining {topic}."
)

summary_prompt = PromptTemplate.from_template(
    "Summarize the following paragraph: {explanation}"
)

#### Building Three Processing Chains

Create three chains that link the prompt templates, the language model, and a string output parser using the pipe operator `(|)` so each component passes its output to the next. 

- Store the first chain in a variable named `name_chain` using the `name_prompt` template. 
- Store the second chain in a variable named `explanation_chain` using the `topic_prompt` template.  
- Store the third chain in a variable named `summary_chain` using the `summary_prompt` template. 

In [3]:
# Create individual chains
name_chain = name_prompt | llm | StrOutputParser()
explanation_chain = topic_prompt | llm | StrOutputParser()
summary_chain = summary_prompt | llm | StrOutputParser()

#### Executing Multiple Chains Sequentially

A function named `run_chains` is defined to take `name` and `topic` parameters, running all three chains sequentially and returning a dictionary containing all three outputs with their corresponding keys.  

- Use `.invoke()` to run the `name_chain` with the `name` input and store the result in a variable named `name_confirm`. 
- Use `.invoke()` to run the `explanation_chain` with the `topic` input and store the result in a variable named `explanation`. 
- Use `.invoke()` to run the `summary_chain` with the `explanation` result and store it in a variable named `summary`. 

In [4]:
# Simple sequential execution function
def run_chains(name, topic):
    """Run the chains sequentially and return results"""
    
    name_confirm = name_chain.invoke({"name": name})
    explanation = explanation_chain.invoke({"topic": topic})
    summary = summary_chain.invoke({"explanation": explanation})
    
    return {
        "name_confirm": name_confirm,
        "explanation": explanation,
        "summary": summary
    }

#### Running and Displaying Chain Results

The `name` and `topic` variables are defined to collect input from the user. 

Call the `run_chains()` function using those `name` and `topic` variables and save the returned dictionary as `result`. 

Run the chains to generate name confirmation, explanation, and summary. The `print()` statements will display all three results. 

In [5]:
# Get user input
name = input("Enter a name: ") 
topic = input("Enter a topic: ")

# Run the sequential chain
print("\n\n\033[1m> Entering new chain...\033[0m")

result = run_chains(name, topic)

print("\n\033[1m> Finished chain.\033[0m")

# Display results
print("\n Name Confirmation:\n", result["name_confirm"])
print("\n Full Explanation:\n", result["explanation"])
print("\n Summary:\n", result["summary"])



> Entering new chain...

> Finished chain.

 Name Confirmation:
 Got it, Nico Figueroa—I’ll remember your name.

 Full Explanation:
 LLMs can track prompts without LangChain memory by using the conversation history included in each request: the model reads the current prompt alongside any earlier messages supplied in its context window and uses them to maintain continuity, follow instructions, refer back to details, and answer follow-up questions. In a basic API setup, however, the model does not automatically remember previous requests; the application must store relevant messages or facts and send them again when making a new request. This works well for short conversations, but the available context is limited, and long histories may need to be summarized or selectively retrieved to avoid exceeding that limit or burying important information. LangChain memory can help organize this process, but it is not required—the application can manage history directly, or a retrieval system c

<details> <summary><strong>Using the Input Box</strong></summary>  

Enter a topic in the input box and press Enter to run the chain. The up and down arrow keys can also be used to navigate through previous inputs. 

#### Testing Memory Limitations

The `followup_prompt` variable is defined with a question asking the LLM if it remembers the user's name.

Invoke the `llm` with this prompt and store the response in the `followup_response` variable. 

The `print()` statement will display the response, demonstrating that the LLM has no memory of the previous interaction. 

In [6]:
# Ask if the LLM remembers your name (it won't)
followup_prompt = "Do you remember my name?"

followup_response = llm.invoke(followup_prompt)

print("\n Follow-up Response:\n", followup_response.content)


 Follow-up Response:
 I don’t see your name in this conversation. What should I call you?


</details> <details> <summary><strong>Note</strong></summary>  

Observe that the name confirmation, explanation, and summary are all generated in a single execution. All three chains run together in one workflow. After the chains finish, try running the follow-up question in Checkpoint 5. Observe that the model cannot remember the name from earlier execution. Each time we run the chain, it treats the input as a completely new request with no memory of previous executions.</details> 